# Fermi-GBM Time-Resolved Spectral Analysis Pipeline
Each cell corresponds to one pipeline stage — run them top to bottom, or re-run any single stage independently.

## Cell 1 — Imports & Setup

In [1]:
import os
import logging
import numpy as np
import matplotlib
matplotlib.use('Agg')   # swap to 'inline' if running interactively
import matplotlib.pyplot as plt
from astropy.table import Table
import astropy.units as u

from threeML import (
    silence_logs, download_GBM_trigger_data, TimeSeriesBuilder,
    OGIPLike, DataList, Model, PointSource, BayesianAnalysis, plot_spectra,
)
from threeML.catalogs import FermiGBMBurstCatalog as FermiGBMBurst
from astromodels import Band, Log_uniform_prior, Truncated_gaussian

silence_logs()
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s  %(levelname)-8s  %(message)s',
    datefmt='%H:%M:%S',
)
log = logging.getLogger('GBM_pipeline')
print('Imports OK')


12:03:02 WARNING   The naima package is not available. Models that depend on it will not be         ]8;id=444434;file:///home/void/anaconda3/envs/threeML/lib/python3.11/site-packages/astromodels/functions/functions_1D/functions.py\functions.py]8;;\:]8;id=263874;file:///home/void/anaconda3/envs/threeML/lib/python3.11/site-packages/astromodels/functions/functions_1D/functions.py#47\47]8;;\
                  available                                                                                        

         WARNING   The GSL library or the pygsl wrapper cannot be loaded. Models that depend on it  ]8;id=943056;file:///home/void/anaconda3/envs/threeML/lib/python3.11/site-packages/astromodels/functions/functions_1D/functions.py\functions.py]8;;\:]8;id=12450;file:///home/void/anaconda3/envs/threeML/lib/python3.11/site-packages/astromodels/functions/functions_1D/functions.py#68\68]8;;\
                  will not be available.                                                                           

         WARNING   The ebltable package is not available. Models that depend on it will not be     ]8;id=514306;file:///home/void/anaconda3/envs/threeML/lib/python3.11/site-packages/astromodels/functions/functions_1D/absorption.py\absorption.py]8;;\:]8;id=977766;file:///home/void/anaconda3/envs/threeML/lib/python3.11/site-packages/astromodels/functions/functions_1D/absorption.py#33\33]8;;\
                  available                                                                                        

12:03:02 INFO      Starting 3ML!                                                                     ]8;id=714553;file:///home/void/anaconda3/envs/threeML/lib/python3.11/site-packages/threeML/__init__.py\__init__.py]8;;\:]8;id=290590;file:///home/void/anaconda3/envs/threeML/lib/python3.11/site-packages/threeML/__init__.py#44\44]8;;\

         WARNING   WARNINGs here are NOT errors                                                      ]8;id=275999;file:///home/void/anaconda3/envs/threeML/lib/python3.11/site-packages/threeML/__init__.py\__init__.py]8;;\:]8;id=655970;file:///home/void/anaconda3/envs/threeML/lib/python3.11/site-packages/threeML/__init__.py#45\45]8;;\

...

         WARNING   Could not import plugin FermiLATLike.py. Do you have the relative instrument     ]8;id=262701;file:///home/void/anaconda3/envs/threeML/lib/python3.11/site-packages/threeML/__init__.py\__init__.py]8;;\:]8;id=432859;file:///home/void/anaconda3/envs/threeML/lib/python3.11/site-packages/threeML/__init__.py#126\126]8;;\
                  software installed and configured?                                                               

         WARNING   No fermitools installed                                              ]8;id=290818;file:///home/void/anaconda3/envs/threeML/lib/python3.11/site-packages/threeML/utils/data_builders/fermi/lat_transient_builder.py\lat_transient_builder.py]8;;\:]8;id=885872;file:///home/void/anaconda3/envs/threeML/lib/python3.11/site-packages/threeML/utils/data_builders/fermi/lat_transient_builder.py#44\44]8;;\

         WARNING   Env. variable MKL_NUM_THREADS is not set. Please set it to 1 for optimal         ]8;id=287449;file:///home/void/anaconda3/envs/threeML/lib/python3.11/site-packages/threeML/__init__.py\__init__.py]8;;\:]8;id=203456;file:///home/void/anaconda3/envs/threeML/lib/python3.11/site-packages/threeML/__init__.py#335\335]8;;\
                  performances in 3ML                                                                              

         WARNING   Env. variable NUMEXPR_NUM_THREADS is not set. Please set it to 1 for optimal     ]8;id=641679;file:///home/void/anaconda3/envs/threeML/lib/python3.11/site-packages/threeML/__init__.py\__init__.py]8;;\:]8;id=661508;file:///home/void/anaconda3/envs/threeML/lib/python3.11/site-packages/threeML/__init__.py#335\335]8;;\
                  performances in 3ML                                                                              

Imports OK


## Cell 2 — Configuration

In [2]:
# ── Edit this block only ────────────────────────────────────────────
CONFIG = dict(
    trigger_name   = 'bn140304557',   # GBM trigger ID  (no 'GRB' prefix)
    grb_name       = 'GRB140304557',  # catalog key      (with 'GRB' prefix)
    ra             = 30.653,           # RA  of the burst [deg]  — update!
    dec            =  33.480,           # Dec of the burst [deg]  — update!

    # detector selection
    max_rov        = 60.0,            # keep detectors with ROV < this [deg]

    # Bayesian-blocks settings
    bb_t_start     = -10.0,           # start of BB window [s relative to trigger]
    bb_t_stop      =  50.0,           # end   of BB window
    bb_p0          =  0.01,           # false-alarm probability for BB
    min_bin_width  =  0.05,           # seconds; narrower bins are merged away

    # spectral energy ranges
    nai_range      = '8-33, 37-900',  # keV  (gap skips K-edge ~34-36 keV)
    bgo_range      = '200-40000',     # keV

    # Band-function priors
    priors = dict(
        K     = dict(lower=1e-4, upper=1e1),
        alpha = dict(lower=-1.7, upper=1.0,  mu=-1.0,  sigma=0.5),
        xp    = dict(lower=10.0, upper=1000, mu=300.0, sigma=200.0),
        beta  = dict(lower=-4.0, upper=-1.8, mu=-2.5,  sigma=1.0),
    ),

    # MultiNest settings
    n_live_points  = 100,             # increase to >=1000 for publication quality

    # output
    outdir         = 'gbm_pipeline_output',
)

# ── Create output directories ─────────────────────────────────────────
PATHS = {
    'root':    CONFIG['outdir'],
    'spectra': os.path.join(CONFIG['outdir'], 'spectra'),
    'corners': os.path.join(CONFIG['outdir'], 'corners'),
    'lc':      os.path.join(CONFIG['outdir'], 'lightcurves'),
}
for d in PATHS.values():
    os.makedirs(d, exist_ok=True)

print('Config loaded. Output directories created at:', CONFIG['outdir'])


Config loaded. Output directories created at: gbm_pipeline_output


## Cell 3 — Helper Function Definitions

In [6]:
def pick_reference_detector(detectors):
    """Prefer n3 as BB reference; fall back to first NaI, then first any."""
    nai = [d for d in detectors if d.startswith('n')]
    if 'n3' in nai: return 'n3'
    return nai[0] if nai else detectors[0]


def build_band_model(ra, dec, priors):
    """Return a fresh Band-function Model with the configured priors."""
    band  = Band()
    model = Model(PointSource('GBM', ra, dec, spectral_shape=band))
    p = priors
    model.GBM.spectrum.main.Band.K.prior = Log_uniform_prior(
        lower_bound=p['K']['lower'], upper_bound=p['K']['upper'])
    model.GBM.spectrum.main.Band.alpha.bounds = (
        p['alpha']['lower'], p['alpha']['upper'])
    model.GBM.spectrum.main.Band.alpha.prior = Truncated_gaussian(
        lower_bound=p['alpha']['lower'], upper_bound=p['alpha']['upper'],
        mu=p['alpha']['mu'], sigma=p['alpha']['sigma'])
    model.GBM.spectrum.main.Band.xp.prior = Truncated_gaussian(
        lower_bound=p['xp']['lower'], upper_bound=p['xp']['upper'],
        mu=p['xp']['mu'], sigma=p['xp']['sigma'])
    model.GBM.spectrum.main.Band.beta.prior = Truncated_gaussian(
        lower_bound=p['beta']['lower'], upper_bound=p['beta']['upper'],
        mu=p['beta']['mu'], sigma=p['beta']['sigma'])
    return model


print('Helper functions defined.')


Helper functions defined.


## Cell 4 — Detector Selection (ROV < 60°)

In [5]:
log.info('Querying Fermi-GBM catalog for %s ...', CONFIG['grb_name'])

cat = FermiGBMBurst()
cat.query_sources(CONFIG['grb_name'])
grb_info = cat.get_detector_information()[CONFIG['grb_name']]

# ── Fetch catalog source / background intervals ───────────────────────
source_interval     = grb_info['source']['fluence']
background_interval = grb_info['background']['full']
log.info('Source interval    : %s', source_interval)
log.info('Background interval: %s', background_interval)

# ── Manually Specify Detectors ────────────────────────────────────────
# Valid NaI detectors: 'n0' through 'n9', 'na', 'nb'
# Valid BGO detectors: 'b0', 'b1'
detectors = ['n3', 'n4', 'b0']  # <-- Change these to your preferred detectors

log.info('Manually selected detectors : %s', detectors)

# ── Quick sanity check ────────────────────────────────────────────────
assert len(detectors) > 0, 'No detectors selected done. Please add them to the list.'
assert any(d.startswith('n') for d in detectors), 'Need at least one NaI detector for the fit.'


12:04:39  INFO      Querying Fermi-GBM catalog for GRB140304557 ...
12:04:42  INFO      Source interval    : -5.376000-25.856000
12:04:42  INFO      Background interval: -40.830000--11.250000,41.670000-97.920000
12:04:42  INFO      Manually selected detectors : ['n3', 'n4', 'b0']


## Cell 5 — Download GBM Data

In [11]:
log.info('Downloading TTE / CSPEC / RSP for: %s', detectors)
dload = download_GBM_trigger_data(CONFIG['trigger_name'], detectors=detectors)

# ── Verify all expected file types are present ────────────────────────
for det in detectors:
    for ftype in ('tte', 'cspec', 'rsp'):
        assert ftype in dload[det], f'Missing {ftype} file for detector {det}'

print('Download complete. Files available for:', list(dload.keys()))


11:26:38  INFO      Downloading TTE / CSPEC / RSP for: ['n3', 'n4', 'b0']


Download complete. Files available for: ['n3', 'n4', 'b0']


## Cell 6 — Build Time Series & Fit Backgrounds

In [12]:
time_series = {}

for det in detectors:
    log.info('Processing detector %s ...', det)

    # ── Step 6a: fit background polynomial on CSPEC ───────────────────
    ts_cspec = TimeSeriesBuilder.from_gbm_cspec_or_ctime(
        det,
        cspec_or_ctime_file = dload[det]['cspec'],
        rsp_file            = dload[det]['rsp'],
    )
    bkg_segs = [s.strip() for s in background_interval.split(',')]
    ts_cspec.set_background_interval(*bkg_segs)

    bkg_file = os.path.join(PATHS['root'], f'{det}_bkg.h5')
    ts_cspec.save_background(bkg_file, overwrite=True)
    log.info('  Background saved to %s', bkg_file)

    # ── Step 6b: load TTE with restored background ────────────────────
    ts_tte = TimeSeriesBuilder.from_gbm_tte(
        det,
        tte_file           = dload[det]['tte'],
        rsp_file           = dload[det]['rsp'],
        restore_background = bkg_file,
    )
    ts_tte.set_active_time_interval(source_interval)
    time_series[det] = ts_tte
    log.info('  TTE loaded and active interval set.')

print('Time series built for:', list(time_series.keys()))

# ── Quick sanity: view lightcurve of first NaI ────────────────────────
first_nai = next(d for d in detectors if d.startswith('n'))
fig = time_series[first_nai].view_lightcurve(-40, 100)
fig.savefig(os.path.join(PATHS['lc'], f'{first_nai}_raw_lc.png'), dpi=100, bbox_inches='tight')
plt.close('all')
print(f'Raw lightcurve saved for {first_nai}')


11:28:11  INFO      Processing detector n3 ...

WARNING RuntimeWarning: invalid value encountered in divide



Loading PHAII Spectra:   0%|                                                                                | …


WARNING RuntimeWarning: invalid value encountered in divide



Finding best polynomial Order:   0%|                                                                          …

Fitting GBM_NAI_03 background:   0%|                                                                         |…

...


WARNING RuntimeWarning: invalid value encountered in divide



Finding best polynomial Order:   0%|                                                                          …

Fitting GBM_BGO_00 background:   0%|                                                                         |…

11:29:16  INFO        Background saved to gbm_pipeline_output/b0_bkg.h5
11:29:16  INFO        TTE loaded and active interval set.


Time series built for: ['n3', 'n4', 'b0']
Raw lightcurve saved for n3


## Cell 7 — Bayesian-Block Time Binning

In [13]:
ref_det = pick_reference_detector(detectors)
ts_ref  = time_series[ref_det]
log.info('Reference detector for BB: %s', ref_det)

# ── Run Bayesian blocks ────────────────────────────────────────────────
ts_ref.create_time_bins(
    CONFIG['bb_t_start'], CONFIG['bb_t_stop'],
    method='bayesblocks', use_background=True, p0=CONFIG['bb_p0'],
)

print('Raw BB bins:')
print('  starts:', ts_ref.bins.starts)
print('  stops :', ts_ref.bins.stops)
print('  widths:', ts_ref.bins.widths)

fig_raw = ts_ref.view_lightcurve(use_binner=True)
fig_raw.savefig(os.path.join(PATHS['lc'], f'{ref_det}_bb_raw.png'), dpi=120, bbox_inches='tight')
plt.close('all')


11:30:11  INFO      Reference detector for BB: n3


Raw BB bins:
  starts: [-9.999761998653412, -6.61941796541214, 7.057295024394989, 11.157336056232452]
  stops : [-6.61941796541214, 7.057295024394989, 11.157336056232452, 49.99896001815796]
  widths: [ 3.38034403 13.67671299  4.10004103 38.84162396]


## Cell 8 — Merge Sub-Threshold Bins

In [14]:
min_w = CONFIG['min_bin_width']

bad_bins = {i for i, w in enumerate(ts_ref.bins.widths) if w < min_w}
print(f'Bins narrower than {min_w} s (to be merged): {sorted(bad_bins)}')

edges = [ts_ref.bins.starts[0]]
for i, b in enumerate(ts_ref.bins):
    if i not in bad_bins:
        edges.append(b.stop)

bb_starts = np.array(edges[:-1])
bb_stops  = np.array(edges[1:])
n_bins    = len(bb_starts)

ts_ref.create_time_bins(bb_starts, bb_stops, method='custom')

print(f'Clean bins ({n_bins} total):')
print('  starts:', bb_starts)
print('  stops :', bb_stops)
print('  widths:', bb_stops - bb_starts)

fig_clean = ts_ref.view_lightcurve(use_binner=True)
fig_clean.savefig(os.path.join(PATHS['lc'], f'{ref_det}_bb_clean.png'), dpi=120, bbox_inches='tight')
plt.close('all')

assert n_bins > 0, 'Zero bins after merging — check bb_t_start / bb_t_stop or p0'
print(f'\nReady to fit {n_bins} time bin(s).')


Bins narrower than 0.05 s (to be merged): []
Clean bins (4 total):
  starts: [-9.999762   -6.61941797  7.05729502 11.15733606]
  stops : [-6.61941797  7.05729502 11.15733606 49.99896002]
  widths: [ 3.38034403 13.67671299  4.10004103 38.84162396]

Ready to fit 4 time bin(s).


## Cell 9 — Sync Detectors & Build Time-Resolved Plugins

In [15]:
time_resolved_plugins = {}   # {det: [plugin_bin_0, plugin_bin_1, ...]}

for det in detectors:
    ts = time_series[det]
    ts.read_bins(ts_ref)                         # sync to reference bins
    plugin_list = ts.to_spectrumlike(from_bins=True)
    time_resolved_plugins[det] = plugin_list
    log.info('%s  ->  %d time-bin plugins', det, len(plugin_list))

# ── Sanity: all detectors should have the same number of bins ─────────
bin_counts = {d: len(v) for d, v in time_resolved_plugins.items()}
print('Plugin counts per detector:', bin_counts)
assert len(set(bin_counts.values())) == 1, 'Bin count mismatch across detectors!'
print('All detectors in sync. Ready for spectral fitting.')


Creating plugins:   0%|                                                                                       …

11:30:56  INFO      n3  ->  4 time-bin plugins


Creating plugins:   0%|                                                                                       …

11:30:56  INFO      n4  ->  4 time-bin plugins


Creating plugins:   0%|                                                                                       …

11:30:57  INFO      b0  ->  4 time-bin plugins


Plugin counts per detector: {'n3': 4, 'n4': 4, 'b0': 4}
All detectors in sync. Ready for spectral fitting.


## Cell 10 — Time-Resolved Spectral Fitting (Band Model)

In [ ]:
PARAM_NAMES = [
    'B_K',     'B_K_p_err',     'B_K_n_err',     'B_K_avg_err',
    'B_alpha', 'B_alpha_n_err', 'B_alpha_p_err',  'B_alpha_avg_err',
    'E_peak',  'E_peak_n_err',  'E_peak_p_err',   'E_peak_avg_err',
    'beta',    'beta_n_err',    'beta_p_err',      'beta_avg_err',
]

fit_rows = []

for i in range(n_bins):
    log.info('==== Time bin %d / %d ====', i + 1, n_bins)

    # ── Assemble per-detector plugins for this bin ────────────────────
    det_plugins = []
    for det in detectors:
        p = time_resolved_plugins[det][i]
        if det.startswith('b'):
            p.set_active_measurements(CONFIG['bgo_range'])
        else:
            ranges = [r.strip() for r in CONFIG['nai_range'].split(',')]
            p.set_active_measurements(*ranges)
        det_plugins.append(p)

    data_list = DataList(*det_plugins)

    # ── Fresh Band model per bin ──────────────────────────────────────
    model = build_band_model(CONFIG['ra'], CONFIG['dec'], CONFIG['priors'])

    # ── Bayesian fit with MultiNest ───────────────────────────────────
    bayes = BayesianAnalysis(model, data_list)
    bayes.set_sampler('multinest')
    bayes.sampler.setup(n_live_points=CONFIG['n_live_points'])
    bayes.sample()

    # ── νFν spectrum plot ─────────────────────────────────────────────
    fig_spec = plot_spectra(
        bayes.results,
        ene_min=1.0, ene_max=1e5, num_ene=200,
        contour_colors='white',
        contour_style_kwargs=dict(alpha=0.2),
        use_components=True, components_to_use=['total'],
        flux_unit='keV2/(cm2 s keV)',
    )
    ax = plt.gca()
    ax.set_xlabel('Energy (keV)', fontsize=16)
    ax.set_ylabel(r'$\nu\,F_\nu\;[\mathrm{keV\,cm^{-2}\,s^{-1}}]$', fontsize=16)
    ax.set_title(f"{CONFIG['trigger_name']}  bin {i+1}/{n_bins}  "
                 f"[{bb_starts[i]:.3f}, {bb_stops[i]:.3f}] s", fontsize=13)
    fig_spec.savefig(
        os.path.join(PATHS['spectra'], f'band_spec_bin{i+1:03d}.png'),
        dpi=120, bbox_inches='tight')
    plt.close('all')

    # ── Corner plot ───────────────────────────────────────────────────
    fig_corner = bayes.results.corner_plot()
    fig_corner.savefig(
        os.path.join(PATHS['corners'], f'corner_bin{i+1:03d}.png'),
        dpi=100, bbox_inches='tight')
    plt.close('all')

    # ── Integrated flux ───────────────────────────────────────────────
    flux = bayes.results.get_point_source_flux(10 * u.keV, 10 * u.MeV)
    log.info('  10 keV-10 MeV flux = %s', flux)

    # ── Collect parameters ────────────────────────────────────────────
    df  = bayes.results.get_data_frame('equal tail')
    df  = df.drop('unit', axis=1, errors='ignore')
    fit_rows.append(df.to_numpy().reshape(16))

    log.info('  Bin %d / %d complete.', i + 1, n_bins)

print(f'\nAll {n_bins} bins fitted successfully.')


11:31:08  INFO      ==== Time bin 1 / 4 ====


 *****************************************************
 MultiNest v3.10
 Copyright Farhan Feroz & Mike Hobson
 Release Jul 2015

 no. of live points =  100
 dimensionality =    4
 *****************************************************
 ln(ev)=  -1262.8796256355897      +/-  0.13212405327503265     
 Total Likelihood Evaluations:          695
 Sampling finished. Exiting MultiNest
  analysing data from chains/fit-.txt


Maximum a posteriori probability (MAP) point:

,result,unit
parameter,,
GBM.spectrum.main.Band.K,(1.02 +0.33 +4) x 10^-4,1 / (keV s cm2)
GBM.spectrum.main.Band.alpha,(-9.4 -2.4 +6) x 10^-1,
GBM.spectrum.main.Band.xp,(4.4 -3.0 +0.5) x 10^2,keV
GBM.spectrum.main.Band.beta,-2.8 -0.7 +0.6,


Values of -log(posterior) at the minimum:

...

Propagating errors:   0%|                                                                                    |…

11:32:30  WARNING   Too few points to create valid contours
11:32:31  WARNING   Too few points to create valid contours
11:32:31  WARNING   Too few points to create valid contours


processing Bayesian analyses:   0%|                                                                           …

11:32:32  INFO        10 keV-10 MeV flux =                                             flux  \
GBM: total  1.1586971268690252e-08 erg / (s cm2)   

                                      low bound  \
GBM: total  6.263700178765745e-09 erg / (s cm2)   

                                        hi bound  
GBM: total  1.7123551581866733e-08 erg / (s cm2)  
11:32:32  INFO        Bin 4 / 4 complete.



All 4 bins fitted successfully.


## Cell 11 — Save Results to CSV

In [17]:
fit_array = np.array(fit_rows)
band_fit_param = Table(fit_array, names=PARAM_NAMES)

# ── Attach bin edges as extra columns ────────────────────────────────
band_fit_param['t_start'] = bb_starts
band_fit_param['t_stop']  = bb_stops
band_fit_param['dt']      = bb_stops - bb_starts

csv_path = os.path.join(PATHS['root'], 'band_fit_params.csv')
band_fit_param.write(csv_path, overwrite=True)

print(f'Results saved to: {csv_path}')
print(f'Shape: {fit_array.shape}  ({n_bins} bins x 16 parameters)')
band_fit_param


Results saved to: gbm_pipeline_output/band_fit_params.csv
Shape: (4, 16)  (4 bins x 16 parameters)


B_K,B_K_p_err,B_K_n_err,B_K_avg_err,B_alpha,B_alpha_p_err,B_alpha_n_err,B_alpha_avg_err,E_peak,E_peak_p_err,E_peak_n_err,E_peak_avg_err,beta,beta_p_err,beta_n_err,beta_avg_err,t_start,t_stop,dt
float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64
0.00010182174118523944,3.2988067919717475e-05,0.0003866710641944488,0.00020982956605708312,-0.9403709372590592,-0.24032793601430158,0.594624307012682,0.4174761215134918,435.7618976974443,-298.63702499471117,46.96982823230178,172.80342661350647,-2.840432810379717,-0.6782810492662916,0.6493053645659224,0.663793206916107,-9.999761998653412,-6.61941796541214,3.380344033241272
0.003493112457683821,-0.001395692145364438,0.0015492200136642898,0.0014724560795143639,-1.0788050761034569,-0.268803087446027,0.2589285371720126,0.2638658123090198,148.9242212776858,-31.291381633546763,145.21568210280495,88.25353186817586,-2.8083400696761878,-0.6381322119912793,0.44842431993589793,0.5432782659635886,-6.61941796541214,7.057295024394989,13.676712989807129
0.007410865594609584,-0.002264477750884203,0.003827554579426992,0.0030460161651555974,-1.1106033788054666,-0.23183165559028618,0.2662424745327947,0.24903706506154044,176.98007810163554,-42.80276966643592,107.93493320067299,75.36885143355445,-2.641981918300892,-0.9256261292302592,0.2577216235759616,0.5916738764031104,7.057295024394989,11.157336056232452,4.100041031837463
0.00011233517020855793,2.6002220613085118e-05,0.0003660577202346743,0.0001960299704238797,-1.378350969583433,-0.06627062580392162,0.6809091991882933,0.37358991249610746,333.0241038294348,-240.33867699858087,78.4598021844335,159.39923959150718,-2.8217900740548485,-0.5984558364290162,0.6222857400050961,0.6103707882170561,11.157336056232452,49.99896001815796,38.84162396192551


## Cell 12 — Pipeline Summary

In [18]:
print('=' * 60)
print('  GBM PIPELINE COMPLETE')
print('=' * 60)
print(f'  Trigger          : {CONFIG["trigger_name"]}')
print(f'  Detectors used   : {detectors}')
print(f'  Time bins fitted : {n_bins}')
print(f'  Bin edges (s)    : starts={bb_starts}, stops={bb_stops}')
print(f'  Output root      : {PATHS["root"]}')
print(f'  Spectra plots    : {PATHS["spectra"]}')
print(f'  Corner plots     : {PATHS["corners"]}')
print(f'  Lightcurves      : {PATHS["lc"]}')
print(f'  CSV results      : {csv_path}')
print('=' * 60)


  GBM PIPELINE COMPLETE
  Trigger          : bn140304557
  Detectors used   : ['n3', 'n4', 'b0']
  Time bins fitted : 4
  Bin edges (s)    : starts=[-9.999762   -6.61941797  7.05729502 11.15733606], stops=[-6.61941797  7.05729502 11.15733606 49.99896002]
  Output root      : gbm_pipeline_output
  Spectra plots    : gbm_pipeline_output/spectra
  Corner plots     : gbm_pipeline_output/corners
  Lightcurves      : gbm_pipeline_output/lightcurves
  CSV results      : gbm_pipeline_output/band_fit_params.csv
